# 关于模型调用的invoke使用

##  1. invoke 可以传入的参数

### 1.1 文本输入

In [42]:
from dotenv import load_dotenv
load_dotenv(override=True)
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain.chat_models import init_chat_model
import os

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model = init_chat_model(
    model="openai:deepseek-flash",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
)

In [43]:
respond = model.invoke("翻译下面的汉字：你好世界")
print(respond)

content='Hello, world.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 44, 'prompt_tokens': 36, 'total_tokens': 80, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 39, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 36}, 'model_provider': 'openai', 'model_name': 'deepseek-flash', 'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6', 'id': 'bf711acf-2410-46f1-89fa-9d8038a435f6', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a19789-f4a6-720d-a819-40ae843f01c7-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 36, 'output_tokens': 44, 'total_tokens': 80, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 39}}


### 1.2 字典列表

In [44]:
messages = [
    {
        "role": "system",
        "content": "你是一个数学老师，最擅长使用言简意赅的方法，为学生讲解不会的数学题。"
    },
    {
        "role": "user",
        "content": "1+1=2么？",
    },
    {
        "role": "assistant",
        "content": "是的，1+1=2，你很聪明",
    },
    {
        "role": "user",
        "content": "什么是歌德巴赫猜想",
    }
]

In [45]:
response = model.invoke(messages)
print(response.content)

哥德巴赫猜想通常指**强哥德巴赫猜想**：

> **任何一个大于 2 的偶数，都可以写成两个素数之和。**

例如：  
4=2+2，6=3+3，8=3+5，10=3+7=5+5。

还有一个**弱哥德巴赫猜想**：

> 任何一个大于 5 的奇数，都可以写成三个素数之和。

弱猜想已被证明；强猜想至今仍未完全证明。陈景润证明的“1+2”就是研究这个问题的重要成果。


**记忆传递**

让模型可以读懂上下文

In [46]:
conversion = [
    {
        "role": "system",
        "content": "你是一个非常友好的AI助手"
    },
    {
        "role": "user",
        "content": "你好，我叫张三",
    },
]

# 第一次对话
response1 = model.invoke(conversion)
print("AI的回复", response1.content)

AI的回复 你好，张三！很高兴认识你 😊 有什么我可以帮你的吗？


In [47]:
# 添加记忆
conversion.append({
        "role": "assistant",
        "content": response1.content
    },
)
conversion.append({
    "role": "user",
    "content": "我叫什么名字？",
},
)

In [48]:
response2 = model.invoke(conversion)
print(response2.content)

你叫张三呀 😊


### 1.3 消息对象列表

In [49]:
messages = [
    SystemMessage(content="你是一个老外，能听懂中文，但是只会说英文"),
    HumanMessage(content="山外青山楼外楼，西湖歌舞几时休的英文是什么？")
]

response = model.invoke(messages)
print(response)

content='英文可以译为：\n\n**“Beyond the hills are more hills; beyond the towers are more towers. When will the singing and dancing by West Lake ever cease?”**\n\n更有诗句感的版本：\n\n**“Hills beyond hills, towers beyond towers; when will West Lake’s songs and dances ever end?”**' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 257, 'prompt_tokens': 59, 'total_tokens': 316, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 194, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 59}, 'model_provider': 'openai', 'model_name': 'deepseek-flash', 'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6', 'id': 'e93af6b1-70aa-4c4d-919a-71296aa9bd22', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--0

**传递记忆**

In [50]:
from langchain_core.messages import AIMessage
messages.append(AIMessage(content=response.content))
messages.append(HumanMessage(content="我刚问了什么问题？"))

In [51]:
response2 = model.invoke(messages)
print(response2.content)

You asked: **“What is the English translation of ‘山外青山楼外楼，西湖歌舞几时休’?”**


## 2. invoke的返回值

In [52]:
response = model.invoke([HumanMessage(content="2+2+3 = ?")])

In [53]:
print(type(response))

<class 'langchain_core.messages.ai.AIMessage'>


In [60]:
# 美化输出方式
from rich import print as rprint
rprint(response)


AIMessage(
    content='7',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 33,
            'prompt_tokens': 37,
            'total_tokens': 70,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': 31,
                'rejected_prediction_tokens': None,
                'text_tokens': None
            },
            'prompt_tokens_details': {
                'audio_tokens': None,
                'cache_write_tokens': None,
                'cached_tokens': 0,
                'image_tokens': None,
                'text_tokens': None
            },
            'prompt_cache_hit_tokens': 0,
            'prompt_cache_miss_tokens': 37
        },
        'model_provider': 'openai',
        'model_name': 'deepseek-flash',
        'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6',
        'id': '7da930c0-790e-4cae-8a1a-b6ed17d8c532',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='lc_run--01a1978b-f9b7-714f-8e7a-e4a993ee3a74-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 37,
        'output_tokens': 33,
        'total_tokens': 70,
        'input_token_details': {'cache_read': 0},
        'output_token_details': {'reasoning': 31}
    }
)